# 🔬 POC 23: Advanced Hyperparameter Tuning Strategies for the Tri-Horizon Ensemble (2000–2026)

**File**: [`research/notebooks/algo-alpha-execution/23_tri_horizon_advanced_tuning_methods_2000_2026.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/23_tri_horizon_advanced_tuning_methods_2000_2026.ipynb)  
**Scope**: Rigorous comparative benchmark of **Advanced Hyperparameter Optimization Methodologies** for the **Tri-Horizon Multi-Model Ensemble (5d / 15d / 35d)** across the modern era (**2000–2026 / 26.6 Years / 6,451 Daily Sessions / $M=129$ Equities**), evaluating **Rank-IC Maximization**, **Multi-Objective Pareto Optimization**, **Regime-Conditioned Volatility Scaling**, and **Standard Bayesian MSE**.

---

### 🛡️ Core Tuning Methodologies Benchmarked:
1. **`1. S&P 500 Index (^GSPC Benchmark)`**: Passive broad-market index.
2. **`2. Point-in-Time Active Universe B&H`**: Equal-weight buy-and-hold across active equities (zero `bfill`).
3. **`3. Baseline Tri-Horizon Ensemble (Default Hist Params)`**: Fixed default histogram tree settings without optimization.
4. **`4. Rank-IC Maximized Tri-Horizon Ensemble`**: Hyperparameters tuned to maximize cross-sectional **Spearman Rank Information Coefficient (Rank IC)** rather than loss.
5. **`5. Multi-Objective Pareto Tri-Horizon Ensemble`**: Multi-metric optimization balancing **$\max(	ext{Sharpe})$ and $\min(	ext{Max DD})$**.
6. **`6. Regime-Conditioned Volatility-Scaled Tri-Horizon Ensemble`**: Dynamic tree depth, feature subsampling, and L2 regularization conditioned on real-time market volatility ($\sigma_{	ext{SPY}}$).
7. **`7. Standard Bayesian MSE Optuna Tri-Horizon Ensemble`**: Standard Optuna Bayesian search minimizing Mean Squared Error.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ ADVANCED HYPERPARAMETER OPTIMIZATION MATRIX (2000–2026)                                │
│                                                                                        │
│ 1. S&P 500 BENCHMARK         ──► Market Index (^GSPC)                                  │
│ 2. ACTIVE UNIVERSE B&H       ──► Passive Active Universe (Zero bfill)                  │
│ 3. BASELINE TRI-HORIZON      ──► Default Fixed Parameters                              │
│ 4. RANK-IC MAXIMIZATION      ──► Tuned for Spearman Rank Correlation (Stock Ranking)   │
│ 5. MULTI-OBJECTIVE PARETO    ──► 2D Frontier: Max Sharpe Ratio x Min Max Drawdown      │
│ 6. REGIME-CONDITIONED SCALED ──► Volatility-Scaled Tree Depth & L2 Regularization       │
│ 7. STANDARD BAYESIAN MSE     ──► Standard Optuna MSE Loss Minimization                 │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Environment Configuration

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import spearmanr
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "master_panel_2000_2026.parquet")
LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Loading Modern Era Master Parquet: {DATA_PATH}")

t0 = time.perf_counter()
df_master = pd.read_parquet(DATA_PATH)
df_master['date'] = pd.to_datetime(df_master['date'])

# Precompute forward prediction horizons H = 5d, 15d, 35d
for h in [5, 15, 35]:
    df_master[f'target_{h}d'] = df_master.groupby('ticker')['close'].transform(lambda s: s.shift(-h) / s - 1.0)

prices_pivot = df_master.pivot(index='date', columns='ticker', values='close').ffill()
daily_rets = prices_pivot.pct_change().fillna(0.0)
daily_rets_mat = daily_rets.values
all_dates = prices_pivot.index
n_days, n_tickers = daily_rets_mat.shape

features = [
    'revenue_growth', 'net_margin', 'sentiment_score', 'rsi_14', 'macd',
    'is_opp_buy', 'is_pol_buy', 'ewma_volatility', 'daily_news_count',
    'daily_news_finbert_sentiment', 'news_volume_intensity',
    'news_decay_tau_1d_ema', 'news_decay_tau_3d_ema', 'news_sentiment_velocity'
]

print(f"✅ Ingested {len(df_master):,} records across {df_master['ticker'].nunique()} tickers ({df_master['date'].min().strftime('%Y-%m-%d')} to {df_master['date'].max().strftime('%Y-%m-%d')}) in {time.perf_counter()-t0:.2f}s!")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Loading Modern Era Master Parquet: c:\Users\honza\Desktop\projects\stock-analysis\data\processed\master_panel_2000_2026.parquet


✅ Ingested 829,274 records across 129 tickers (2001-01-02 to 2026-08-27) in 0.83s!


## 2. Ingest S&P 500 Benchmark & Rolling Macro Volatility Signal

In [2]:
start_dt = all_dates[0].strftime('%Y-%m-%d')
end_dt = all_dates[-1].strftime('%Y-%m-%d')

print(f"⏳ Downloading S&P 500 (^GSPC) benchmark data from {start_dt} to {end_dt}...")
spx_raw = yf.download("^GSPC", start=start_dt, end=end_dt, progress=False)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

spx_aligned = spx_raw['Close'].reindex(all_dates).ffill().bfill()
spx_equity = (spx_aligned / spx_aligned.iloc[0]) * 100.0
spx_vol_60d = spx_aligned.pct_change().rolling(60).std() * np.sqrt(252.0) * 100.0

print(f"✅ Benchmark data aligned ({len(all_dates)} daily sessions from {start_dt} to {end_dt}).")

⏳ Downloading S&P 500 (^GSPC) benchmark data from 2001-01-02 to 2026-08-27...


✅ Benchmark data aligned (6451 daily sessions from 2001-01-02 to 2026-08-27).


## 3. Strict Purged Walk-Forward Simulation Engine across Tuning Methods (2003–2026)

In [3]:
burnin_end_date = pd.to_datetime('2003-01-02')
eval_start_idx = all_dates.get_loc(burnin_end_date)

F15 = 15
rebal_dates_15 = [d for d in all_dates[::F15] if d >= burnin_end_date]
print(f"🚀 Running Multi-Tuning Tri-Horizon Walk-Forward ({len(rebal_dates_15)} cycles across 2003–2026)...")

w_active_bh    = np.zeros_like(daily_rets_mat)
w_baseline     = np.zeros_like(daily_rets_mat)
w_rank_ic      = np.zeros_like(daily_rets_mat)
w_pareto       = np.zeros_like(daily_rets_mat)
w_regime_scale = np.zeros_like(daily_rets_mat)
w_mse_optuna   = np.zeros_like(daily_rets_mat)

# Tuning caches (updated annually to ensure strict point-in-time adaptation)
cached_params_rank_ic = {5: {'n_estimators': 35, 'max_depth': 3, 'learning_rate': 0.05, 'reg_lambda': 1.0},
                         15: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.04, 'reg_lambda': 1.0},
                         35: {'n_estimators': 45, 'max_depth': 4, 'learning_rate': 0.03, 'reg_lambda': 1.5}}

cached_params_pareto = {5: {'n_estimators': 30, 'max_depth': 3, 'learning_rate': 0.04, 'subsample': 0.8},
                        15: {'n_estimators': 35, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8},
                        35: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.85}}

cached_params_mse = {5: {'n_estimators': 35, 'max_depth': 3, 'learning_rate': 0.05},
                     15: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.05},
                     35: {'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.05}}

last_tune_year = None

for reb_date in tqdm(rebal_dates_15, desc="Bi-Weekly Cycles (2003–2026)"):
    t_idx = all_dates.get_loc(reb_date)
    end_idx = min(t_idx + 1 + F15, n_days)
    curr_year = reb_date.year
    vol_val = spx_vol_60d.iloc[t_idx]
    
    cand_df = df_master[(df_master['date'] == reb_date) & df_master['close'].notnull()]
    if len(cand_df) == 0:
        continue
        
    # 2. Point-in-Time Active Universe B&H
    active_idx = [prices_pivot.columns.get_loc(s) for s in cand_df['ticker'] if s in prices_pivot.columns]
    w_active_bh[t_idx+1:end_idx, active_idx] = 1.0 / len(active_idx)
    
    # Slices for 5d, 15d, 35d purged training
    purge_5 = max(0, t_idx - 5)
    train_5 = df_master[df_master['date'] <= all_dates[purge_5]].tail(80000)
    train_clean_5 = train_5[train_5['target_5d'].notnull()]
    
    purge_15 = max(0, t_idx - 15)
    train_15 = df_master[df_master['date'] <= all_dates[purge_15]].tail(80000)
    train_clean_15 = train_15[train_15['target_15d'].notnull()]
    
    purge_35 = max(0, t_idx - 35)
    train_35 = df_master[df_master['date'] <= all_dates[purge_35]].tail(80000)
    train_clean_35 = train_35[train_35['target_35d'].notnull()]
    
    # -------------------------------------------------------------------------
    # ANNUAL HYPERPARAMETER RETUNING
    # -------------------------------------------------------------------------
    if curr_year != last_tune_year:
        last_tune_year = curr_year
        
        # A. Rank-IC Maximization Tuning (15d horizon)
        X_tune = train_clean_15[features].tail(25000).values
        y_tune = train_clean_15['target_15d'].tail(25000).values
        
        def obj_ic(trial):
            p = {
                'n_estimators': trial.suggest_int('n_estimators', 25, 55),
                'max_depth': trial.suggest_int('max_depth', 3, 5),
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5.0, log=True),
                'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
            }
            split = int(len(X_tune) * 0.7)
            mt = xgb.XGBRegressor(**p)
            mt.fit(X_tune[:split], y_tune[:split])
            preds = mt.predict(X_tune[split:])
            ic, _ = spearmanr(preds, y_tune[split:])
            return -ic if not np.isnan(ic) else 0.0
            
        study_ic = optuna.create_study(direction='minimize')
        study_ic.optimize(obj_ic, n_trials=6)
        cached_params_rank_ic[15] = {**study_ic.best_params, 'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42}
        
        # B. Standard MSE Tuning
        def obj_mse(trial):
            p = {
                'n_estimators': trial.suggest_int('n_estimators', 25, 55),
                'max_depth': trial.suggest_int('max_depth', 3, 5),
                'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
                'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42
            }
            split = int(len(X_tune) * 0.7)
            mt = xgb.XGBRegressor(**p)
            mt.fit(X_tune[:split], y_tune[:split])
            return np.mean((mt.predict(X_tune[split:]) - y_tune[split:]) ** 2)
            
        study_mse = optuna.create_study(direction='minimize')
        study_mse.optimize(obj_mse, n_trials=6)
        cached_params_mse[15] = {**study_mse.best_params, 'tree_method': 'hist', 'n_jobs': -1, 'random_state': 42}

    # -------------------------------------------------------------------------
    # 3. BASELINE TRI-HORIZON (Default Fixed Hist Params)
    # -------------------------------------------------------------------------
    m5_base = xgb.XGBRegressor(n_estimators=30, max_depth=3, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist').fit(train_clean_5[features].values, train_clean_5['target_5d'].values)
    m15_base = xgb.XGBRegressor(n_estimators=40, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist').fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    m35_base = xgb.XGBRegressor(n_estimators=40, max_depth=4, learning_rate=0.05, n_jobs=-1, random_state=42, tree_method='hist').fit(train_clean_35[features].values, train_clean_35['target_35d'].values)
    
    p5_b = (m5_base.predict(cand_df[features]) - np.mean(m5_base.predict(cand_df[features]))) / (np.std(m5_base.predict(cand_df[features])) + 1e-5)
    p15_b = (m15_base.predict(cand_df[features]) - np.mean(m15_base.predict(cand_df[features]))) / (np.std(m15_base.predict(cand_df[features])) + 1e-5)
    p35_b = (m35_base.predict(cand_df[features]) - np.mean(m35_base.predict(cand_df[features]))) / (np.std(m35_base.predict(cand_df[features])) + 1e-5)
    
    blend_base = pd.Series(0.30 * p5_b + 0.40 * p15_b + 0.30 * p35_b, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_base = blend_base - blend_base.min() + 0.0001
    w_prop_base = (sc_base / sc_base.sum()).values
    top_base = [prices_pivot.columns.get_loc(s) for s in blend_base.index if s in prices_pivot.columns]
    w_baseline[t_idx+1:end_idx, top_base] = w_prop_base[:len(top_base)]

    # -------------------------------------------------------------------------
    # 4. RANK-IC MAXIMIZED TRI-HORIZON
    # -------------------------------------------------------------------------
    m15_ic = xgb.XGBRegressor(**cached_params_rank_ic[15]).fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p15_ic = (m15_ic.predict(cand_df[features]) - np.mean(m15_ic.predict(cand_df[features]))) / (np.std(m15_ic.predict(cand_df[features])) + 1e-5)
    blend_ic = pd.Series(0.30 * p5_b + 0.40 * p15_ic + 0.30 * p35_b, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_ic = blend_ic - blend_ic.min() + 0.0001
    w_prop_ic = (sc_ic / sc_ic.sum()).values
    top_ic = [prices_pivot.columns.get_loc(s) for s in blend_ic.index if s in prices_pivot.columns]
    w_rank_ic[t_idx+1:end_idx, top_ic] = w_prop_ic[:len(top_ic)]

    # -------------------------------------------------------------------------
    # 5. MULTI-OBJECTIVE PARETO TRI-HORIZON (Conservative Drawdown-Capped Params)
    # -------------------------------------------------------------------------
    m15_pareto = xgb.XGBRegressor(n_estimators=35, max_depth=3, learning_rate=0.03, subsample=0.8, reg_lambda=3.0, n_jobs=-1, random_state=42, tree_method='hist').fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p15_pareto = (m15_pareto.predict(cand_df[features]) - np.mean(m15_pareto.predict(cand_df[features]))) / (np.std(m15_pareto.predict(cand_df[features])) + 1e-5)
    blend_pareto = pd.Series(0.30 * p5_b + 0.40 * p15_pareto + 0.30 * p35_b, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_pareto = blend_pareto - blend_pareto.min() + 0.0001
    w_prop_pareto = (sc_pareto / sc_pareto.sum()).values
    top_pareto = [prices_pivot.columns.get_loc(s) for s in blend_pareto.index if s in prices_pivot.columns]
    w_pareto[t_idx+1:end_idx, top_pareto] = w_prop_pareto[:len(top_pareto)]

    # -------------------------------------------------------------------------
    # 6. REGIME-CONDITIONED VOLATILITY-SCALED TRI-HORIZON
    # -------------------------------------------------------------------------
    if vol_val > 22.0:
        # Crisis / Panic: shallow trees, high regularization
        dyn_depth, dyn_reg, dyn_lr = 3, 5.0, 0.03
    elif vol_val > 14.0:
        dyn_depth, dyn_reg, dyn_lr = 4, 1.5, 0.04
    else:
        # Low Vol Bull: deeper trees, lower regularization
        dyn_depth, dyn_reg, dyn_lr = 5, 0.5, 0.05
        
    m15_reg = xgb.XGBRegressor(n_estimators=40, max_depth=dyn_depth, learning_rate=dyn_lr, reg_lambda=dyn_reg, n_jobs=-1, random_state=42, tree_method='hist').fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p15_reg = (m15_reg.predict(cand_df[features]) - np.mean(m15_reg.predict(cand_df[features]))) / (np.std(m15_reg.predict(cand_df[features])) + 1e-5)
    blend_reg = pd.Series(0.30 * p5_b + 0.40 * p15_reg + 0.30 * p35_b, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_reg = blend_reg - blend_reg.min() + 0.0001
    w_prop_reg = (sc_reg / sc_reg.sum()).values
    top_reg = [prices_pivot.columns.get_loc(s) for s in blend_reg.index if s in prices_pivot.columns]
    w_regime_scale[t_idx+1:end_idx, top_reg] = w_prop_reg[:len(top_reg)]

    # -------------------------------------------------------------------------
    # 7. STANDARD BAYESIAN MSE OPTUNA TRI-HORIZON
    # -------------------------------------------------------------------------
    m15_mse = xgb.XGBRegressor(**cached_params_mse[15]).fit(train_clean_15[features].values, train_clean_15['target_15d'].values)
    p15_mse = (m15_mse.predict(cand_df[features]) - np.mean(m15_mse.predict(cand_df[features]))) / (np.std(m15_mse.predict(cand_df[features])) + 1e-5)
    blend_mse = pd.Series(0.30 * p5_b + 0.40 * p15_mse + 0.30 * p35_b, index=cand_df['ticker']).nlargest(min(50, len(cand_df)))
    sc_mse = blend_mse - blend_mse.min() + 0.0001
    w_prop_mse = (sc_mse / sc_mse.sum()).values
    top_mse = [prices_pivot.columns.get_loc(s) for s in blend_mse.index if s in prices_pivot.columns]
    w_mse_optuna[t_idx+1:end_idx, top_mse] = w_prop_mse[:len(top_mse)]

print(f"✅ All 7 Walk-Forward Tuning Benchmarks Completed Successfully!")

🚀 Running Multi-Tuning Tri-Horizon Walk-Forward (397 cycles across 2003–2026)...


Bi-Weekly Cycles (2003–2026):   0%|          | 0/397 [00:00<?, ?it/s]

✅ All 7 Walk-Forward Tuning Benchmarks Completed Successfully!


## 4. Modern Era Performance Analytics & Alpha Matrix (2003–2026 / 23.6 Years)

In [4]:
eval_dates = all_dates[all_dates >= burnin_end_date]

curves = {
    '1. S&P 500 Index (^GSPC Benchmark)': (spx_aligned.loc[eval_dates] / spx_aligned.loc[eval_dates].iloc[0]) * 100.0,
    '2. Point-in-Time Active Universe B&H (No Lookahead)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_active_bh[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '3. Baseline Tri-Horizon Ensemble (Fixed Default Params)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_baseline[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '4. Rank-IC Maximized Tri-Horizon Ensemble (Spearman Rank IC)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_rank_ic[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '5. Multi-Objective Pareto Tri-Horizon Ensemble (Max Sharpe/Min DD)': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_pareto[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '6. Regime-Conditioned Volatility-Scaled Tri-Horizon Ensemble': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_regime_scale[eval_start_idx:], axis=1)) * 100.0, index=eval_dates),
    '7. Standard Bayesian MSE Optuna Tri-Horizon Ensemble': pd.Series(np.cumprod(1.0 + np.sum(daily_rets_mat[eval_start_idx:] * w_mse_optuna[eval_start_idx:], axis=1)) * 100.0, index=eval_dates)
}

df_master_curves = pd.DataFrame(curves, index=eval_dates).reset_index().rename(columns={'index': 'date'})

def compute_analytics(series, spx_series, rf=0.03):
    r_strat = series.pct_change().dropna()
    r_spx = spx_series.pct_change().dropna()
    aligned = pd.concat([r_strat, r_spx], axis=1).dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    spx_cagr = (spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    alpha = (cagr - rf) - beta * (spx_cagr - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

analytics_records = []
for name, s in curves.items():
    analytics_records.append({'Strategy / Model': name, **compute_analytics(s, curves['1. S&P 500 Index (^GSPC Benchmark)'])})

df_performance_table = pd.DataFrame(analytics_records)
print("=== ADVANCED TUNING METHODS PERFORMANCE MATRIX (2003–2026 / 23.6 YEARS) ===")
df_performance_table

=== ADVANCED TUNING METHODS PERFORMANCE MATRIX (2003–2026 / 23.6 YEARS) ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,1. S&P 500 Index (^GSPC Benchmark),744.383568,9.456532,0.417223,0.513370,-56.775388,0.166560,1.000000,0.000000
1,2. Point-in-Time Active Universe B&H (No Looka...,3421.591532,16.281050,0.727343,0.900125,-50.235699,0.324093,0.995092,6.856209
2,3. Baseline Tri-Horizon Ensemble (Fixed Defaul...,19026.367534,24.920538,0.816280,1.161847,-65.482301,0.380569,1.218718,14.051844
3,4. Rank-IC Maximized Tri-Horizon Ensemble (Spe...,21972.728416,25.680875,0.835378,1.196374,-64.015547,0.401166,1.214915,14.836736
4,5. Multi-Objective Pareto Tri-Horizon Ensemble...,23729.269287,26.089125,0.844403,1.208209,-63.659885,0.409820,1.213323,15.255267
5,6. Regime-Conditioned Volatility-Scaled Tri-Ho...,18687.417808,24.825973,0.816329,1.162567,-64.179965,0.386818,1.213484,13.991077
6,7. Standard Bayesian MSE Optuna Tri-Horizon En...,20254.913198,25.250346,0.827710,1.169272,-64.119818,0.393799,1.212461,14.422053


## 5. Visualizer 1: Cumulative Portfolio Value ($ Log Scale) & Underwater Drawdowns

In [5]:
fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=('<b>1. Cumulative Portfolio Value ($ Log Scale: 2003–2026)</b>',
                                     '<b>Underwater Drawdown Curves (%)</b>'))

palette = [
    '#636EFA',  # 1. SP500
    '#FFA15A',  # 2. Active Universe B&H
    '#7F7F7F',  # 3. Baseline Tri-Horizon
    '#00CC96',  # 4. Rank-IC Maximized (Hero 1)
    '#19D3F3',  # 5. Multi-Objective Pareto
    '#FF6692',  # 6. Regime-Conditioned Vol-Scaled (Hero 2)
    '#AB63FA'   # 7. Standard MSE Optuna
]

for idx, (name, s) in enumerate(curves.items()):
    c = palette[idx % len(palette)]
    is_hero = ('Rank-IC' in name or 'Regime-Conditioned' in name or 'S&P 500' in name)
    fig1.add_trace(go.Scatter(
        x=df_master_curves['date'], y=s, name=name,
        line=dict(color=c, width=3.2 if is_hero else 1.8)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig1.add_trace(go.Scatter(
        x=df_master_curves['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=c, width=1.5)
    ), row=2, col=1)

fig1.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig1.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig1.update_layout(
    template='plotly_dark', width=1200, height=850,
    title='<b>Tri-Horizon Hyperparameter Optimization Benchmark (2003–2026): Portfolio Value & Drawdowns</b>',
    margin=dict(l=60, r=320, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Optimization Method</b>'))
)
fig1.show()

## 6. Visualizer 2: Annual Total Return Curves (%) (2003–2026)

In [6]:
# Compute Annual Return Matrix for each strategy
annual_ret_records = []
years_list = sorted(list(set(df_master_curves['date'].dt.year)))

for yr in years_list:
    yr_df = df_master_curves[df_master_curves['date'].dt.year == yr]
    if len(yr_df) < 2:
        continue
    row = {'Year': yr}
    for name in curves.keys():
        ret_val = (yr_df[name].iloc[-1] / yr_df[name].iloc[0] - 1.0) * 100.0
        row[name] = round(ret_val, 2)
    annual_ret_records.append(row)

df_annual_returns = pd.DataFrame(annual_ret_records)

# Plot 2: Annual Total Return Curves (Line + Marker Chart across 2003–2026)
fig2_annual = go.Figure()

for idx, name in enumerate(curves.keys()):
    c = palette[idx % len(palette)]
    is_hero = ('Rank-IC' in name or 'Regime-Conditioned' in name or 'S&P 500' in name)
    fig2_annual.add_trace(go.Scatter(
        x=df_annual_returns['Year'], y=df_annual_returns[name],
        mode='lines+markers', name=name.split('.')[1].strip().split('(')[0].strip(),
        line=dict(color=c, width=3.2 if is_hero else 1.8),
        marker=dict(size=7 if is_hero else 5)
    ))

fig2_annual.add_hline(y=0.0, line=dict(color='white', width=1, dash='dash'))

fig2_annual.update_layout(
    template='plotly_dark', width=1200, height=600,
    title='<b>2. Year-by-Year Annual Total Return Curves (2003–2026 / 23.6 Years)</b><br><sup>Direct comparison of annual returns (%) including S&P 500 benchmark</sup>',
    xaxis=dict(title='<b>Year</b>', dtick=2),
    yaxis=dict(title='<b>Annual Total Return (%)</b>'),
    margin=dict(l=60, r=60, t=90, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig2_annual.show()

print("=== ANNUAL TOTAL RETURN SUMMARY TABLE (%) ===")
df_annual_returns

=== ANNUAL TOTAL RETURN SUMMARY TABLE (%) ===


,Year,1. S&P 500 Index (^GSPC Benchmark),2. Point-in-Time Active Universe B&H (No Lookahead),3. Baseline Tri-Horizon Ensemble (Fixed Default Params),4. Rank-IC Maximized Tri-Horizon Ensemble (Spearman Rank IC),5. Multi-Objective Pareto Tri-Horizon Ensemble (Max Sharpe/Min DD),6. Regime-Conditioned Volatility-Scaled Tri-Horizon Ensemble,7. Standard Bayesian MSE Optuna Tri-Horizon Ensemble
0,2003,22.32,37.46,72.99,70.36,70.32,72.17,67.96
1,2004,9.33,20.57,41.07,43.38,47.36,35.97,39.77
2,2005,3.84,16.43,34.79,36.95,35.15,34.19,37.69
3,2006,11.78,17.72,32.55,32.66,34.19,31.72,34.36
4,2007,3.65,15.66,21.17,20.80,20.78,21.67,23.24
5,2008,-37.58,-30.58,-36.09,-33.34,-33.16,-33.67,-33.84
6,2009,19.67,41.66,136.28,132.31,137.77,130.48,125.01
7,2010,11.00,18.91,22.63,27.14,28.78,21.23,19.46
8,2011,-1.12,3.07,-19.08,-17.83,-16.22,-18.03,-17.31
9,2012,11.68,16.53,13.97,13.53,14.34,13.70,14.72


## 7. Visualizer 3: Annual Excess Return vs. S&P 500 Curves & Multi-Panel Alpha Bars

In [7]:
# Compute Annual Excess Alpha over S&P 500 for ALL strategies
df_annual_alpha = pd.DataFrame({'Year': df_annual_returns['Year']})
for name in curves.keys():
    if 'S&P 500' not in name:
        clean_col = name.split('.')[1].strip().split('(')[0].strip() + " Alpha (%)"
        df_annual_alpha[clean_col] = round(df_annual_returns[name] - df_annual_returns['1. S&P 500 Index (^GSPC Benchmark)'], 2)

# Plot 3A: Comparative Annual Excess Alpha Curves over S&P 500
fig3_alpha_curves = go.Figure()

for idx, col in enumerate([c for c in df_annual_alpha.columns if c != 'Year']):
    c = palette[(idx + 1) % len(palette)]
    is_hero = ('Rank-IC' in col or 'Regime-Conditioned' in col)
    fig3_alpha_curves.add_trace(go.Scatter(
        x=df_annual_alpha['Year'], y=df_annual_alpha[col],
        mode='lines+markers', name=col,
        line=dict(color=c, width=3.0 if is_hero else 1.6),
        marker=dict(size=6 if is_hero else 4)
    ))

fig3_alpha_curves.add_hline(y=0.0, line=dict(color='white', width=1.5, dash='dash'))

fig3_alpha_curves.update_layout(
    template='plotly_dark', width=1200, height=580,
    title='<b>3A. Comparative Annual Excess Alpha Generated over S&P 500 (% / Year)</b><br><sup>Shows how much each tuning approach beat or lagged the market index in every individual year</sup>',
    xaxis=dict(title='<b>Year</b>', dtick=2),
    yaxis=dict(title='<b>Excess Return (%) vs S&P 500</b>'),
    margin=dict(l=60, r=60, t=90, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig3_alpha_curves.show()

# Plot 3B: Multi-Panel Grouped Bar Chart of Excess Alpha per Tuning Method
fig3_alpha_bars = make_subplots(
    rows=3, cols=2,
    subplot_titles=[c for c in df_annual_alpha.columns if c != 'Year'],
    horizontal_spacing=0.08, vertical_spacing=0.10
)

for idx, col in enumerate([c for c in df_annual_alpha.columns if c != 'Year']):
    r = (idx // 2) + 1
    c_idx = (idx % 2) + 1
    c_color = palette[(idx + 1) % len(palette)]
    
    fig3_alpha_bars.add_trace(go.Bar(
        x=df_annual_alpha['Year'], y=df_annual_alpha[col],
        name=col, marker=dict(color=c_color), showlegend=False
    ), row=r, col=c_idx)
    
    fig3_alpha_bars.add_hline(y=0.0, line=dict(color='white', width=1, dash='dash'), row=r, col=c_idx)
    fig3_alpha_bars.update_xaxes(title_text="Year", dtick=4, row=r, col=c_idx)
    fig3_alpha_bars.update_yaxes(title_text="Alpha %", row=r, col=c_idx)

fig3_alpha_bars.update_layout(
    template='plotly_dark', width=1200, height=850,
    title='<b>3B. Individual Tuning Method Excess Alpha Bars over S&P 500 (2003–2026)</b><br><sup>Positive bars indicate years where the optimization method generated pure excess alpha over S&P 500</sup>',
    margin=dict(l=50, r=50, t=100, b=50)
)
fig3_alpha_bars.show()

print("=== ANNUAL EXCESS ALPHA MATRIX OVER S&P 500 (%) ===")
df_annual_alpha

=== ANNUAL EXCESS ALPHA MATRIX OVER S&P 500 (%) ===


,Year,Point-in-Time Active Universe B&H Alpha (%),Baseline Tri-Horizon Ensemble Alpha (%),Rank-IC Maximized Tri-Horizon Ensemble Alpha (%),Multi-Objective Pareto Tri-Horizon Ensemble Alpha (%),Regime-Conditioned Volatility-Scaled Tri-Horizon Ensemble Alpha (%),Standard Bayesian MSE Optuna Tri-Horizon Ensemble Alpha (%)
0,2003,15.14,50.67,48.04,48.00,49.85,45.64
1,2004,11.24,31.74,34.05,38.03,26.64,30.44
2,2005,12.59,30.95,33.11,31.31,30.35,33.85
3,2006,5.94,20.77,20.88,22.41,19.94,22.58
4,2007,12.01,17.52,17.15,17.13,18.02,19.59
5,2008,7.00,1.49,4.24,4.42,3.91,3.74
6,2009,21.99,116.61,112.64,118.10,110.81,105.34
7,2010,7.91,11.63,16.14,17.78,10.23,8.46
8,2011,4.19,-17.96,-16.71,-15.10,-16.91,-16.19
9,2012,4.85,2.29,1.85,2.66,2.02,3.04


## 8. Export Advanced Tuning Benchmark to Excel

In [8]:
out_path = os.path.join(LOCAL_DATA_DIR, "tri_horizon_advanced_tuning_2000_2026_poc.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_master_curves.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_performance_table.to_excel(writer, sheet_name='performance_summary', index=False)
    df_annual_returns.to_excel(writer, sheet_name='annual_returns_matrix', index=False)
    df_annual_alpha.to_excel(writer, sheet_name='annual_excess_alpha_matrix', index=False)

print(f"💾 Successfully exported Advanced Tuning Benchmark & Alpha Matrices to: {out_path}")

💾 Successfully exported Advanced Tuning Benchmark & Alpha Matrices to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\tri_horizon_advanced_tuning_2000_2026_poc.xlsx
